# 7a. Organoid QC

## Purpose
Flag low-quality organoids per patient using two criteria applied in sequence:
1. **NaN detection** — organoids missing key metadata or feature values
2. **Size outliers** — abnormally small or large organoids by volume (z-score)

This is **step 7a of Stage 4 (image-based profiling)**. It runs once per patient
and must complete before `7b.single_cell_qc.ipynb`, which inherits organoid flags.

## Inputs
- `data/{patient}/image_based_profiles/3.annotated_profiles/organoid_anno.parquet`

## Outputs
- `data/{patient}/image_based_profiles/4.qc_profiles/organoid_flagged_outliers.parquet`
  — original organoid profile with three added `Metadata_cqc_*` flag columns

## Notes
- QC flags are **additive**: an organoid can be flagged by multiple criteria simultaneously.
- Outlier detection only runs on the subset of organoids that passed the NaN check,
  so NaN rows are never evaluated for size outliers.

In [1]:
import os
import pathlib

import pandas as pd
from cosmicqc import find_outliers
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
# NOTE: previously this line unconditionally overrode bandicoot_check()
# with root_dir, meaning bandicoot was never actually used even when
# mounted. Removed so bandicoot_check()'s own bandicoot-first behavior
# takes effect.

In [2]:
if not in_notebook:
    args = parse_args()
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    image_based_profiles_subparent_name = "image_based_profiles"
    patient = "NF0037_T1_CQ1"

## Load in all the organoid profiles and concat together

In [3]:
organoid_file = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles"
    / "organoid_anno.parquet"
).resolve(strict=True)

sammed_annotated_organoid_profiles_path = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles"
    / "sammed_organoid_anno.parquet"
).resolve()

qc_output_dir = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "4.qc_profiles"
)
qc_output_dir.mkdir(parents=True, exist_ok=True)

organoid_qc_output_path = f"{qc_output_dir}/organoid_flagged_outliers.parquet"
sammed_organoid_qc_output_path = (
    f"{qc_output_dir}/sammed_organoid_flagged_outliers.parquet"
)

orig_organoid_profiles_df = pd.read_parquet(organoid_file)

# Print the shape and head of the combined organoid profiles DataFrame
print(orig_organoid_profiles_df.shape)
orig_organoid_profiles_df.head()

(1837, 909)


,Metadata_Biology_PatientID,Metadata_Biology_PatientTumor,Metadata_Biology_TumorType,Metadata_Compartment,Metadata_Experiment_Class,Metadata_Experiment_Dose,Metadata_Experiment_ImageSet,Metadata_Experiment_PlateID,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,...,Organoid_NoChannel_VolumeSizeShape_EulerNumber,Organoid_NoChannel_VolumeSizeShape_Extent,Organoid_NoChannel_VolumeSizeShape_MaxX,Organoid_NoChannel_VolumeSizeShape_MaxY,Organoid_NoChannel_VolumeSizeShape_MaxZ,Organoid_NoChannel_VolumeSizeShape_MinX,Organoid_NoChannel_VolumeSizeShape_MinY,Organoid_NoChannel_VolumeSizeShape_MinZ,Organoid_NoChannel_VolumeSizeShape_SurfaceArea,Organoid_NoChannel_VolumeSizeShape_Volume
0,NF0037,NF0037_T1_CQ1,cNF,Organoid,Small Molecule,1,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,...,1.0,0.656185,1179.0,972.0,40.0,832.0,713.0,0.0,3707.606685,23589.34
1,NF0037,NF0037_T1_CQ1,cNF,Organoid,Small Molecule,1,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,...,1.0,0.598096,1361.0,1530.0,8.0,1070.0,1297.0,1.0,581.270843,2838.69
2,NF0037,NF0037_T1_CQ1,cNF,Organoid,Control,1,NF0037_T1_CQ1__NF0037_T1_CQ1__F9__F9,NF0037_T1_CQ1,Control,Control,...,1.0,0.584435,1126.0,1169.0,39.0,593.0,807.0,0.0,5922.005486,43978.11
3,NF0037,NF0037_T1_CQ1,cNF,Organoid,Small Molecule,1,NF0037_T1_CQ1__NF0037_T1_CQ1__D7__F13,NF0037_T1_CQ1,receptor tyrosine kinase inhibitor,Kinase Inhibitor,...,1.0,0.588361,1530.0,1222.0,40.0,1254.0,971.0,0.0,3341.240701,16303.71
4,NF0037,NF0037_T1_CQ1,cNF,Organoid,Small Molecule,1,NF0037_T1_CQ1__NF0037_T1_CQ1__D7__F13,NF0037_T1_CQ1,receptor tyrosine kinase inhibitor,Kinase Inhibitor,...,1.0,0.754918,1099.0,1240.0,36.0,922.0,1091.0,1.0,1745.816897,6968.31


## Round 1 QC: flag rows with NaN in key columns

`Metadata_cqc_*` columns are boolean flags added by this notebook. A value of `True`
means the organoid failed that criterion. Multiple flags can be True simultaneously.

We flag organoids where `ObjectID`, `SingleCellCount`, or `Volume` is NaN because:
- An organoid with no cells (`SingleCellCount` NaN) cannot be a valid profile row.
- A NaN `ObjectID` means the object does not exist and all features will be NaN.
- A NaN `Volume` means the core morphology feature is missing.

In [4]:
organoid_profiles_df = orig_organoid_profiles_df.copy()
organoid_profiles_df["Metadata_cqc_nan_detected"] = (
    organoid_profiles_df[
        [
            "Metadata_Object_ObjectID",
            "Metadata_Object_OrganoidSingleCellCount",
            "Organoid_NoChannel_VolumeSizeShape_Volume",
        ]
    ]
    .isna()
    .any(axis=1)
)
# Print the number of organoids flagged
flagged_count = organoid_profiles_df["Metadata_cqc_nan_detected"].sum()
print(f"Number of organoids flagged: {flagged_count}")

organoid_profiles_df.head()

Number of organoids flagged: 0


,Metadata_Biology_PatientID,Metadata_Biology_PatientTumor,Metadata_Biology_TumorType,Metadata_Compartment,Metadata_Experiment_Class,Metadata_Experiment_Dose,Metadata_Experiment_ImageSet,Metadata_Experiment_PlateID,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,...,Organoid_NoChannel_VolumeSizeShape_Extent,Organoid_NoChannel_VolumeSizeShape_MaxX,Organoid_NoChannel_VolumeSizeShape_MaxY,Organoid_NoChannel_VolumeSizeShape_MaxZ,Organoid_NoChannel_VolumeSizeShape_MinX,Organoid_NoChannel_VolumeSizeShape_MinY,Organoid_NoChannel_VolumeSizeShape_MinZ,Organoid_NoChannel_VolumeSizeShape_SurfaceArea,Organoid_NoChannel_VolumeSizeShape_Volume,Metadata_cqc_nan_detected
0,NF0037,NF0037_T1_CQ1,cNF,Organoid,Small Molecule,1,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,...,0.656185,1179.0,972.0,40.0,832.0,713.0,0.0,3707.606685,23589.34,False
1,NF0037,NF0037_T1_CQ1,cNF,Organoid,Small Molecule,1,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,...,0.598096,1361.0,1530.0,8.0,1070.0,1297.0,1.0,581.270843,2838.69,False
2,NF0037,NF0037_T1_CQ1,cNF,Organoid,Control,1,NF0037_T1_CQ1__NF0037_T1_CQ1__F9__F9,NF0037_T1_CQ1,Control,Control,...,0.584435,1126.0,1169.0,39.0,593.0,807.0,0.0,5922.005486,43978.11,False
3,NF0037,NF0037_T1_CQ1,cNF,Organoid,Small Molecule,1,NF0037_T1_CQ1__NF0037_T1_CQ1__D7__F13,NF0037_T1_CQ1,receptor tyrosine kinase inhibitor,Kinase Inhibitor,...,0.588361,1530.0,1222.0,40.0,1254.0,971.0,0.0,3341.240701,16303.71,False
4,NF0037,NF0037_T1_CQ1,cNF,Organoid,Small Molecule,1,NF0037_T1_CQ1__NF0037_T1_CQ1__D7__F13,NF0037_T1_CQ1,receptor tyrosine kinase inhibitor,Kinase Inhibitor,...,0.754918,1099.0,1240.0,36.0,922.0,1091.0,1.0,1745.816897,6968.31,False


## Process non-NaN rows to detect abnormally small and large organoids and flag them

In [5]:
# Set the metadata columns to be used in the QC process
metadata_columns = [x for x in organoid_profiles_df.columns if "Metadata" in x]

In [6]:
## Round 2 QC: size-based outlier detection

# `find_outliers` uses z-score thresholds: negative values flag objects below the mean,
# positive values flag objects above. Threshold magnitude is the number of standard
# deviations from the mean. Only non-NaN rows (from Round 1) are evaluated.

# Only process the rows that are not flagged
filtered_profile_df = organoid_profiles_df[
    ~organoid_profiles_df["Metadata_cqc_nan_detected"]
]

# Find outlier organoids based on the 'Volume.Size.Shape_Organoid_VOLUME' column
print("Finding small organoid outliers...")
small_size_outliers = find_outliers(
    df=filtered_profile_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Organoid_NoChannel_VolumeSizeShape_Volume": -1,  # Detect very small organoids
    },
)

# Ensure the column exists before assignment
organoid_profiles_df["Metadata_cqc_small_organoid_outlier"] = False
organoid_profiles_df.loc[
    small_size_outliers.index, "Metadata_cqc_small_organoid_outlier"
] = True

print("Finding large organoid outliers...")
large_size_outliers = find_outliers(
    df=filtered_profile_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Organoid_NoChannel_VolumeSizeShape_Volume": 3,  # Detect very large organoids
    },
)

# Ensure the column exists before assignment
organoid_profiles_df["Metadata_cqc_large_organoid_outlier"] = False
organoid_profiles_df.loc[
    large_size_outliers.index, "Metadata_cqc_large_organoid_outlier"
] = True

# Print number of outliers (only in filtered rows)
small_count = filtered_profile_df.index.intersection(small_size_outliers.index).shape[0]
large_count = filtered_profile_df.index.intersection(large_size_outliers.index).shape[0]
print(f"Small organoid outliers found: {small_count}")
print(f"Large organoid outliers found: {large_count}")

organoid_profiles_df.to_parquet(organoid_qc_output_path, index=False)

Finding small organoid outliers...
Number of outliers: 0 (0.00%)
Outliers Range:
Organoid_NoChannel_VolumeSizeShape_Volume Min: nan
Organoid_NoChannel_VolumeSizeShape_Volume Max: nan
Finding large organoid outliers...
Number of outliers: 47 (2.56%)
Outliers Range:
Organoid_NoChannel_VolumeSizeShape_Volume Min: 91861.03000000001
Organoid_NoChannel_VolumeSizeShape_Volume Max: 217100.41000000003
Small organoid outliers found: 0
Large organoid outliers found: 47


In [7]:
# Print example output of the flagged organoid profiles
print(organoid_profiles_df.shape)
organoid_profiles_df.head()

(1837, 912)


,Metadata_Biology_PatientID,Metadata_Biology_PatientTumor,Metadata_Biology_TumorType,Metadata_Compartment,Metadata_Experiment_Class,Metadata_Experiment_Dose,Metadata_Experiment_ImageSet,Metadata_Experiment_PlateID,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,...,Organoid_NoChannel_VolumeSizeShape_MaxY,Organoid_NoChannel_VolumeSizeShape_MaxZ,Organoid_NoChannel_VolumeSizeShape_MinX,Organoid_NoChannel_VolumeSizeShape_MinY,Organoid_NoChannel_VolumeSizeShape_MinZ,Organoid_NoChannel_VolumeSizeShape_SurfaceArea,Organoid_NoChannel_VolumeSizeShape_Volume,Metadata_cqc_nan_detected,Metadata_cqc_small_organoid_outlier,Metadata_cqc_large_organoid_outlier
0,NF0037,NF0037_T1_CQ1,cNF,Organoid,Small Molecule,1,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,...,972.0,40.0,832.0,713.0,0.0,3707.606685,23589.34,False,False,False
1,NF0037,NF0037_T1_CQ1,cNF,Organoid,Small Molecule,1,NF0037_T1_CQ1__NF0037_T1_CQ1__B9__F13,NF0037_T1_CQ1,HDAC inhibitor,Histone Deacetylase Inhibitors,...,1530.0,8.0,1070.0,1297.0,1.0,581.270843,2838.69,False,False,False
2,NF0037,NF0037_T1_CQ1,cNF,Organoid,Control,1,NF0037_T1_CQ1__NF0037_T1_CQ1__F9__F9,NF0037_T1_CQ1,Control,Control,...,1169.0,39.0,593.0,807.0,0.0,5922.005486,43978.11,False,False,False
3,NF0037,NF0037_T1_CQ1,cNF,Organoid,Small Molecule,1,NF0037_T1_CQ1__NF0037_T1_CQ1__D7__F13,NF0037_T1_CQ1,receptor tyrosine kinase inhibitor,Kinase Inhibitor,...,1222.0,40.0,1254.0,971.0,0.0,3341.240701,16303.71,False,False,False
4,NF0037,NF0037_T1_CQ1,cNF,Organoid,Small Molecule,1,NF0037_T1_CQ1__NF0037_T1_CQ1__D7__F13,NF0037_T1_CQ1,receptor tyrosine kinase inhibitor,Kinase Inhibitor,...,1240.0,36.0,922.0,1091.0,1.0,1745.816897,6968.31,False,False,False


## Merge the qc flags to the deep learning-based profiles and save the output
Merge the QC flags back to the original organoid profiles, which will be used in downstream analyses and single cell QC. 
We need to do this beacuase we do not run qc on black-box features. 
Merge on the Metadata_Biology_PatientTumor, Metadata_Experiment_WellFOV
and the Metadata_Object_ObjectID columns, which together uniquely identify each organoid profile row.

In [8]:
sammed_organoid_df = pd.read_parquet(sammed_annotated_organoid_profiles_path)
original_sammed_shape = sammed_organoid_df.shape
# set the merge keys to int for both dataframes to ensure they match
merge_keys = [
    "Metadata_Biology_PatientTumor",
    "Metadata_Experiment_WellFOV",
    "Metadata_Object_ObjectID",
]
qc_keys = [col for col in organoid_profiles_df.columns if "Metadata_cqc" in col]


# merge the flagged organoid profiles with the sammed annotated organoid profiles
qc_annotated_sammed_organoid_df = sammed_organoid_df.merge(
    organoid_profiles_df[qc_keys + merge_keys],
    on=merge_keys,
    how="left",
)
if qc_annotated_sammed_organoid_df.shape[1] == original_sammed_shape[1]:
    raise ValueError(
        f"No new columns were added during the merge. Check that the merge keys {merge_keys} are correct and that the qc keys {qc_keys} are present in the organoid_profiles_df."
    )
qc_annotated_sammed_organoid_df.to_parquet(sammed_organoid_qc_output_path, index=False)
qc_annotated_sammed_organoid_df.head()

,Metadata_Biology_PatientTumor,Metadata_Biology_TumorType,Metadata_Experiment_Class,Metadata_Experiment_Dose,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Unit,Metadata_Experiment_ViabilityPercentage,Metadata_Experiment_Well,...,Organoid_Mito_SAMMed3D_Feature-global93,Organoid_Mito_SAMMed3D_Feature-global94,Organoid_Mito_SAMMed3D_Feature-global95,Organoid_Mito_SAMMed3D_Feature-global96,Organoid_Mito_SAMMed3D_Feature-global97,Organoid_Mito_SAMMed3D_Feature-global98,Organoid_Mito_SAMMed3D_Feature-global99,Metadata_cqc_nan_detected,Metadata_cqc_small_organoid_outlier,Metadata_cqc_large_organoid_outlier
0,NF0037_T1_CQ1,cNF,Small Molecule,1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,uM,85.253054,B9,...,-0.010997,0.054389,-0.037063,0.164367,0.031119,0.288731,0.045690,NaN,NaN,NaN
1,NF0037_T1_CQ1,cNF,Small Molecule,1,HDAC inhibitor,Histone Deacetylase Inhibitors,Panobinostat,uM,85.253054,B9,...,-0.010963,0.053950,-0.036204,0.168216,0.021030,0.289422,0.035523,NaN,NaN,NaN
2,NF0037_T1_CQ1,cNF,Control,1,Control,Control,DMSO,%,NaN,F9,...,-0.010911,0.054128,-0.026074,0.165265,0.041558,0.266512,0.059619,NaN,NaN,NaN
3,NF0037_T1_CQ1,cNF,Small Molecule,1,receptor tyrosine kinase inhibitor,Kinase Inhibitor,Cabozantinib,uM,88.132635,D7,...,-0.011159,0.053444,-0.038601,0.162831,0.011732,0.288139,0.028453,NaN,NaN,NaN
4,NF0037_T1_CQ1,cNF,Small Molecule,1,receptor tyrosine kinase inhibitor,Kinase Inhibitor,Cabozantinib,uM,88.132635,D7,...,-0.011031,0.052032,-0.027650,0.163275,0.034445,0.276816,0.049330,NaN,NaN,NaN
